# 第2章章节实践：OpenMP 扩展性

## 本节学习目标

本实践要求综合运用本章知识完成可复现的工程任务。请保留命令、参数、正确性结果和分析结论。

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
import os, platform, shutil
print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("CMake:", shutil.which("cmake"))
print("OMP_NUM_THREADS:", os.getenv("OMP_NUM_THREADS", "not set"))


## 必要背景与实验材料

本实践基于本章 `src/` 中的课程工程副本。开始前应完成前面各小节，并能解释工程的关键源码、构建入口、正确性门槛和计时字段。

## 实践任务

1. 通过 OMP_NUM_THREADS 比较环境允许的多个线程数
2. 通过 OMP_SCHEDULE 在保持其他变量不变时比较 static 与另一种 schedule
3. 记录正确性、耗时、speedup、并行效率
4. 解释带宽饱和、负载不均和调度开销

## 核心知识与关键源码解析

本章实践只调整 Notebook 中的运行参数和实验配置，不修改 `src/`。请保持输入与正确性阈值一致，每次只改变一个变量，并说明它如何影响数据流和性能。

## 实验记录模板

课程提供 6 个矩阵：U1/U2 为行长度较均匀矩阵，L1/L2 为长尾矩阵，B1/B2 为块状矩阵。先用 U1 观察均匀负载下的调度开销，再用 L1 观察长行集中时的负载差异。执行下面的命令并记录结果。

In [ ]:
%%bash
set -e
cd src/openmp_spmv
bash scripts/build.sh
for threads in 1 2 4; do
  OMP_NUM_THREADS=$threads OMP_SCHEDULE=static \
    bash scripts/run.sh --matrix U1 --warmup 1 --repeat 3 \
    --csv "results/U1_${threads}t_static.csv"
done
# U1：4 线程 static 已由上面的循环生成；这里只改变调度策略。
OMP_NUM_THREADS=4 OMP_SCHEDULE=dynamic,16 \
  bash scripts/run.sh --matrix U1 --warmup 1 --repeat 3 \
  --csv results/U1_4t_dynamic16.csv

# L1：保持矩阵、线程数和重复次数一致，只改变调度策略。
OMP_NUM_THREADS=4 OMP_SCHEDULE=static \
  bash scripts/run.sh --matrix L1 --warmup 1 --repeat 3 \
  --csv results/L1_4t_static.csv
OMP_NUM_THREADS=4 OMP_SCHEDULE=dynamic,16 \
  bash scripts/run.sh --matrix L1 --warmup 1 --repeat 3 \
  --csv results/L1_4t_dynamic16.csv

## 评价标准

必须展示唯一修改变量和正确性检查；不得把参考 CSV 当作本人结果，结论应区分调度开销、负载不均和带宽饱和。

## 查看参考答案

参考答案给出方法和判断依据，不提供虚构的固定性能数字。

## 预期现象与结果分析

正确性门槛应首先通过；性能结果随硬件、软件栈和系统负载变化。若修改后没有加速或出现退化，也应依据阶段计时、通信次数或资源竞争给出解释。

## 实践小结

完成报告时，应明确实验环境、唯一修改变量、正确性门槛、计时口径和观察到的限制。

## 工程实践提交物与完成标准

章测必须基于 `src/openmp_spmv/`，不得只回答概念题。操作链：构建 → 跑串行/OpenMP 基线 → 设置 `OMP_NUM_THREADS=1,2,4,8,16` → 设置 `OMP_SCHEDULE=static,dynamic,guided` → 固定输入与 repeat → 记录并分析。

提交物：实际命令与环境；阅读或修改的真实文件/函数/参数；字段为“Threads、Schedule、CPU single、OpenMP、Speedup、Error”的结果表；正确性判据；基于数据的结论。性能数字不作为固定答案。

完成标准：命令指向真实脚本或可执行文件，数据来自同口径运行，并能解释结果。


## 四类考核

以下四题中，客观题答案唯一，凭 CSR 的 `row_ptr` 区间语义即可判定；简单/中等/困难题基于本章实验，要求用命令、输出、CSV 或计算过程作为证据，不接受无证据的概念回答。

### 1. 客观题

（1）单选：CSR 的 `row_ptr` 长度为 rows+1，其中 `row_ptr[rows]` 的值恒等于（　）

A. 0　　B. rows　　C. nnz　　D. 最后一行的行号

（2）判断（对/错）：按行 `parallel for` 并行 SpMV 时，第 r 行只写 `y[r]`，任意两行 r1≠r2 的写目标下标不同，因此天然无写冲突，不需要原子操作；x 的读取可能重叠，但只读不写，也不需要同步。（　）

### 2. 简单题

给出线程数单变量实验的完整命令与结果表：固定矩阵、schedule 与 repeat，运行 `OMP_NUM_THREADS=1,2,4,8`（环境允许时含 16），表格字段为 Threads、CPU single、OpenMP、Speedup、Error；说明串行 reference 来自哪一次运行、误差门槛是多少。

### 3. 中等题

分别对 U1 和 L1 保持矩阵、线程数和 repeat 不变，只把 `OMP_SCHEDULE` 从 static 改为 dynamic,16。给出四次运行的 CSV/输出，比较均匀行与长尾行下的调度开销和负载差异；结论必须引用实测字段。

### 4. 困难题

用实测数据判断平台期：计算随线程数变化的 speedup、并行效率和每线程内存带宽（按读入 value/col_idx 与间接 x 的字节量估算），指出带宽饱和出现的线程数，并区分 bandwidth-bound、负载不均与调度开销各自的表现证据。
